In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
import sys
from pathlib import Path

project_root = Path.cwd()
if not (project_root / "src").is_dir():
    project_root = project_root.parent

sys.path.insert(0, str(project_root))

from src.train_model import XGBoost
from src.train_model import Logistic_Regression
from src.train_model import Random_Forest
from sklearn.preprocessing import OneHotEncoder

In [2]:
df = pd.read_csv(r"C:\Academics\vs code\Projects\Customer-Churn-Prediction\data\dataset.csv.csv")

In [3]:
df.isna().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [4]:
df.describe()

,SeniorCitizen,tenure,MonthlyCharges
count,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692
std,0.368612,24.559481,30.090047
min,0.000000,0.000000,18.250000
25%,0.000000,9.000000,35.500000
50%,0.000000,29.000000,70.350000
75%,0.000000,55.000000,89.850000
max,1.000000,72.000000,118.750000


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [6]:
df.drop(columns=['customerID'], inplace=True)

In [7]:
df.head(5)

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [8]:
col = ['Contract', 'PaymentMethod', 'InternetService', 'MultipleLines']
df = pd.get_dummies(df, columns=col, drop_first=True)

In [9]:
df.head(5)

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,...,Churn,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,InternetService_Fiber optic,InternetService_No,MultipleLines_No phone service,MultipleLines_Yes
0,Female,0,Yes,No,1,No,No,Yes,No,No,...,No,False,False,False,True,False,False,False,True,False
1,Male,0,No,No,34,Yes,Yes,No,Yes,No,...,No,True,False,False,False,True,False,False,False,False
2,Male,0,No,No,2,Yes,Yes,Yes,No,No,...,Yes,False,False,False,False,True,False,False,False,False
3,Male,0,No,No,45,No,Yes,No,Yes,Yes,...,No,True,False,False,False,False,False,False,True,False
4,Female,0,No,No,2,Yes,No,No,No,No,...,Yes,False,False,False,True,False,True,False,False,False


In [10]:
one = OneHotEncoder()
one.fit_transform(df)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 176075 stored elements and shape (7043, 8239)>

In [11]:
le = LabelEncoder()
for i in df.columns:
    df[i] = le.fit_transform(df[i])


In [12]:
df.head(5)

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,...,Churn,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,InternetService_Fiber optic,InternetService_No,MultipleLines_No phone service,MultipleLines_Yes
0,0,0,1,0,1,0,0,2,0,0,...,0,0,0,0,1,0,0,0,1,0
1,1,0,0,0,34,1,2,0,2,0,...,0,1,0,0,0,1,0,0,0,0
2,1,0,0,0,2,1,2,2,0,0,...,1,0,0,0,0,1,0,0,0,0
3,1,0,0,0,45,0,2,0,2,2,...,0,1,0,0,0,0,0,0,1,0
4,0,0,0,0,2,1,0,0,0,0,...,1,0,0,0,1,0,1,0,0,0


In [13]:
y = df['Churn']

In [14]:
x = df.drop(columns=['Churn'])

In [15]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2)

In [16]:
sc = StandardScaler()
x_train = sc.fit_transform(x_train)
x_test = sc.transform(x_test)

In [17]:
model1 , accuracy1 = Logistic_Regression(x_train, x_test, y_train, y_test)

In [18]:
model2, accuracy2 = Random_Forest(x_train, x_test, y_train, y_test)

In [19]:
model3, accuracy3 = XGBoost(x_train, x_test, y_train, y_test)

In [21]:
print("Accuracy of Logistic_Regression is : ", accuracy1)
print("Accuracy of Random_Forest is : ", accuracy2)
print("Accuracy of XGBoost is : ", accuracy3)


Accuracy of Logistic_Regression is :  80.7
Accuracy of Random_Forest is :  79.63
Accuracy of XGBoost is :  80.62
